# 04 — Audit Check
**Purpose:** verify the latest pipeline run is healthy; flag drift exceeding threshold.
**Reads:** bronze_yellow_taxi, silver_yellow_taxi, rejected_records, pipeline_log
**Writes:** one row to pipeline_log with stage = "audit_check"      
**Decisions:**
- Reconciliation: bronze == silver + rejected + structural_drops (hard requirement; raises if not)
- Drift threshold: structural_drops / bronze must be ≤ 5% (soft; logged as WARN if exceeded)

In [0]:
%run ./00_utils

In [0]:
import uuid
RUN_ID = str(uuid.uuid4())

Loaded helpers: LOG_TABLE, LOG_SCHEMA, log_pipeline_run, PIPELINE_NAME


### Hard vs soft checks

Reconciliation is a **hard check** — we `raise` if `bronze ≠ silver + rejected + structural_drops` because that math failure means data went missing somewhere, and downstream consumers should not see broken state.

Drift threshold is a **soft check** — exceeding 5% logs a `WARN` to the audit table but doesn't block. The two checks separate "this pipeline is mathematically broken" from "this pipeline ingested data of unusually low quality."

The 5% threshold is intentionally configurable. In production it would be derived from a rolling 30-day baseline rather than a hardcoded constant — sudden 2x spikes in drift are the real signal, not the absolute number.

In [0]:
BRONZE_TABLE   = "workspace.taxi.bronze_yellow_taxi"
SILVER_TABLE   = "workspace.taxi.silver_yellow_taxi"
REJECTED_TABLE = "workspace.taxi.rejected_records"
DRIFT_THRESHOLD_PCT = 5.0

try:
    bronze_count   = spark.table(BRONZE_TABLE).count()
    silver_count   = spark.table(SILVER_TABLE).count()
    rejected_count = spark.table(REJECTED_TABLE).count()

    structural_drops = bronze_count - silver_count - rejected_count
    drift_pct        = (structural_drops / bronze_count) * 100.0 if bronze_count else 0.0
    rejection_pct    = (rejected_count   / bronze_count) * 100.0 if bronze_count else 0.0

    print("=== Pipeline health snapshot ===")
    print(f"Bronze:                  {bronze_count:>12,}")
    print(f"Silver (clean):          {silver_count:>12,}")
    print(f"Rejected (quarantined):  {rejected_count:>12,}  ({rejection_pct:.2f}%)")
    print(f"Structural drops:        {structural_drops:>12,}  ({drift_pct:.2f}%)")
    print(f"Drift threshold:         {DRIFT_THRESHOLD_PCT}%")

    # Hard check: reconciliation must hold
    accounted = silver_count + rejected_count + structural_drops
    if accounted != bronze_count:
        raise ValueError(
            f"RECONCILIATION FAILED: silver({silver_count}) + rejected({rejected_count}) "
            f"+ structural_drops({structural_drops}) = {accounted} != bronze({bronze_count})"
        )

    # Soft check: drift threshold
    if drift_pct > DRIFT_THRESHOLD_PCT:
        status = "WARN"
        msg = f"Structural drop rate {drift_pct:.2f}% exceeds threshold {DRIFT_THRESHOLD_PCT}%"
        print(f"\n⚠ {msg}")
    else:
        status = "SUCCESS"
        msg = f"Drift {drift_pct:.2f}% within threshold; rejection rate {rejection_pct:.2f}%"
        print(f"\n✓ {msg}")

    log_pipeline_run("audit_check", bronze_count, silver_count, status, msg)

except Exception as e:
    log_pipeline_run("audit_check", -1, 0, "FAILED", str(e))
    raise

=== Pipeline health snapshot ===
Bronze:                     2,964,624
Silver (clean):             2,756,127
Rejected (quarantined):        68,335  (2.31%)
Structural drops:             140,162  (4.73%)
Drift threshold:         5.0%

✓ Drift 4.73% within threshold; rejection rate 2.31%


/home/spark-340c53d3-2d16-42cd-aafa-84/.ipykernel/2090/command-7449222514757034-4272143697:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(PIPELINE_NAME, RUN_ID, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],


[audit_check] SUCCESS | rows_in=2,964,624 rows_out=2,756,127


In [0]:
print("=== Full pipeline_log (chronological) ===")
spark.sql(f"""
    SELECT stage,
           rows_in,
           rows_out,
           status,
           CASE WHEN length(error_message) > 60
                THEN concat(substring(error_message, 1, 57), '...')
                ELSE error_message
           END AS error_message,
           run_timestamp
    FROM {LOG_TABLE}
    ORDER BY run_timestamp ASC
""").show(truncate=False)

=== Full pipeline_log (chronological) ===
+-----------+-------+--------+-------+------------------------------------------------------------+--------------------------+
|stage      |rows_in|rows_out|status |error_message                                               |run_timestamp             |
+-----------+-------+--------+-------+------------------------------------------------------------+--------------------------+
|bronze     |-1     |0       |FAILED |[CANNOT_DETERMINE_TYPE] Some of types cannot be determine...|2026-05-05 10:07:09.445778|
|bronze     |2964624|2964624 |SUCCESS|NULL                                                        |2026-05-05 10:08:12.485299|
|silver     |2964624|2824462 |SUCCESS|NULL                                                        |2026-05-05 10:33:17.289635|
|silver     |2964624|2756127 |SUCCESS|NULL                                                        |2026-05-05 10:36:15.762095|
|silver     |2964624|2756127 |SUCCESS|NULL                           